# p_1 — 09 · the experiments: the actual science

**The research question:** *does grounding reduce hallucination in a model we
built from scratch?* — plus three questions any honest measurement needs
answered first.

| # | experiment | what it answers |
|---|------------|-----------------|
| 1 | **Myth contamination A/B** | Train Config B **twice** — clean `corpus/health_qa.jsonl` vs contaminated `corpus/corpus_with_myths.jsonl` (+300 planted myth pairs) — with **identical seed (42) and schedule**. Measure the myth-parroting rate (eval_lib's `mythVerdict`) of BOTH brains on the 20 `eval_mythtraps.jsonl` traps. *Does poisoning the corpus move the parroting rate?* |
| 2 | **Grounding delta** (the headline) | With the CLEAN model: 3 eval sets × 4 modes (RAW / GROUNDED / VERIFIED / ABSTAINING) through `eval_lib`. *How much does grounding cut the hallucination rate?* |
| 3 | **Abstention threshold sweep** | Sweep the refusal threshold tau over 12 values (0.2×–1.5× the Task 9 calibrated median). True refusals on unanswerables vs false refusals on answerables → precision/recall/**F1** → pick the operating point and compare with Task 9's hardcoded `ABSTAIN_TAU`. *Is the default threshold actually the best trade?* |
| 4 | **Seed variance** | Train the **baby** config with 3 seeds (42/7/1234) on the clean corpus; measure RAW vs GROUNDED hallucination per seed → mean ± spread. *How noisy are these metrics — how much should we trust any single number?* |

**How to run**
1. `LOCAL_SMOKE = True` (below): tiny configs, few steps — every code path
   runs end-to-end in **minutes** on CPU. The numbers prove plumbing, not
   science.
2. `LOCAL_SMOKE = False` in Colab: the real battery (Config B arms @ 6,000
   steps ≈ 4.5–5 h **per arm** on free-tier CPU + baby @ 800 steps/seed).
   Checkpoints + payloads make every section **resumable** — re-running a
   cell reuses finished work instead of recomputing it.

**Honesty rules baked in:** each experiment prints its own `EXPn DONE` line;
a failure in one experiment prints a visible traceback and the battery
*continues*; `results/summary.md` renders "(did not complete)" for anything
missing and marks stub runs `STUB` in the provenance. No number is invented.

Matplotlib is allowed but not needed here — the tables **are** the result.

## 0 · Environment — Drive guard, paths, imports

In [ ]:
# ---- environment: Drive, paths, repo (same guards as 04/06/08) ----
import os, sys, json, pathlib, traceback
IN_COLAB = 'google.colab' in str(sys.modules or {})
try:
    from google.colab import drive  # noqa: F401
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    pass

if IN_COLAB:
    ROOT = pathlib.Path('/content/drive/MyDrive/p_1')   # corpus/ checkpoints/ results/
    REPO = pathlib.Path('/content/p_1')                 # code: run 00_git_setup.ipynb first
else:
    ROOT = pathlib.Path('.').resolve()
    REPO = ROOT

sys.path.insert(0, str(REPO / 'colab'))
for sub in ('corpus', 'checkpoints', 'results'):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

CORPUS_DIR = ROOT / 'corpus'
RESULTS    = ROOT / 'results'
CKPT_ROOT  = ROOT / 'checkpoints' / 'experiments'      # one subdir per arm/seed

# retrieval_py / train_lib resolve data via P1_DATA_DIR > Colab Drive > repo
if (CORPUS_DIR / 'health_qa.jsonl').exists():
    os.environ['P1_DATA_DIR'] = str(CORPUS_DIR)
print('ROOT      :', ROOT)
print('REPO      :', REPO)
print('IN_COLAB  :', IN_COLAB)
print('checkpoints:', CKPT_ROOT)

In [ ]:
# ---- the one flag that matters: LOCAL_SMOKE -----------------------------
#   True  → tiny configs + few steps. Proves every CODE PATH in minutes.
#           The numbers are near-babble plumbing, NOT science.
#   False → the real battery: Config B arms @ 6,000 steps (04_train contract,
#           ~4.5–5 h per arm on Colab CPU — resumable) + baby @ 800 steps/seed.
LOCAL_SMOKE = True

import experiments_lib as xp          # the reusable logic (colab/experiments_lib.py)

MODE          = 'smoke' if LOCAL_SMOKE else 'full'
MODEL_CFG     = xp.model_config(MODE)     # E1/E2 arms (full = CONFIG_B)
BABY_CFG      = xp.baby_config(MODE)      # E4 baby (full = Config A dims, BPE vocab)
SCHED         = xp.schedule(MODE)         # lr / warmup / batch — identical within each pair
STEPS_AB      = xp.steps_for(MODE, 'ab')
STEPS_SEED    = xp.steps_for(MODE, 'seed')
SEEDS         = xp.SEEDS                  # (42, 7, 1234)
SEED_AB       = xp.BASE_SEED              # 42 — IDENTICAL for both A/B arms
TAU           = xp.el.ABSTAIN_TAU         # Task 9 calibrated refusal threshold
MAX_NEW_TOKENS = xp.MAX_NEW_TOKENS        # 80 (the 08_eval contract)

print('mode            :', MODE)
print('E1/E2 model cfg :', MODEL_CFG)
print('E4 baby cfg     :', BABY_CFG)
print('schedule        :', SCHED, '· steps AB=%d seed=%d' % (STEPS_AB, STEPS_SEED))
print('seeds           :', list(SEEDS), '· A/B seed =', SEED_AB)
print('tau             : %.4f · max new tokens: %d' % (TAU, MAX_NEW_TOKENS))
if LOCAL_SMOKE:
    print('\n(LOCAL_SMOKE: expect ~5–10 min total on a small CPU box)')

In [ ]:
# ---- corpora, eval sets, retrieval index, tau continuity ----
import eval_lib as el
import retrieval_py as rp

corpora = {'clean': CORPUS_DIR / 'health_qa.jsonl',
           'contaminated': CORPUS_DIR / 'corpus_with_myths.jsonl'}
sets = {name: rp.load_jsonl(CORPUS_DIR / ('eval_%s.jsonl' % name))
        for name in ('answerable', 'mythtraps', 'unanswerable')}
n_clean = len(rp.load_jsonl(corpora['clean']))
n_cont  = len(rp.load_jsonl(corpora['contaminated']))
print('corpora      : clean %d pairs · contaminated %d pairs (+%d planted myth pairs)'
      % (n_clean, n_cont, n_cont - n_clean))
print('eval sets    :', {k: len(v) for k, v in sets.items()})

index = rp.build_index(rp.load_jsonl(corpora['clean']))
print('index        : N=%d chunks · vocab=%d · avgdl=%.3f · build %.0f ms'
      % (index['N'], index['vocab'], index['avgdl'], index['build_ms']))

# Task 9 continuity: the hardcoded ABSTAIN_TAU must re-derive from THIS index
cal = el.calibrate_tau(index, sets['answerable'])
assert abs(cal - el.ABSTAIN_TAU) < 5e-4, (cal, el.ABSTAIN_TAU)
print('tau check    : calibrated median %.6f == hardcoded %.4f (Task 9 contract)'
      % (cal, el.ABSTAIN_TAU))

## 1 · Experiment 1 — myth contamination A/B

**Design.** Two siblings, identical in every way except the training corpus:

| arm | corpus | pairs |
|-----|--------|-------|
| clean | `health_qa.jsonl` | 4,278 mainstream-guidance Q&A |
| contaminated | `corpus_with_myths.jsonl` | same + **300 planted myth-affirming pairs** (folk beliefs asserted as fact, e.g. "Sure it does…" — the seed corpus Task 2 built for exactly this experiment) |

Same init seed (42), same lr/warmup/steps (`total_steps` = the schedule
target, so the cosine decay is identical too), same data-pack seed — the
ONLY difference is what the siblings read. Both are then quizzed on the 20
`eval_mythtraps.jsonl` folk-belief traps ("Does cracking your knuckles cause
arthritis?") and the RAW answers are scored by `eval_lib.mythVerdict`:
the answer's mean content-F1 against the **folk belief** vs the
**correction**; parroting = folk wins by more than the myth margin (with a
correction-cue override: an answer containing "no evidence / not true /
actually…" can't count as parroting).

**The science:** if the contaminated sibling's parroting rate moves above the
clean sibling's, corpus poisoning measurably changed what the model asserts.

*(In LOCAL_SMOKE both brains are near-babble after ~150 tiny steps — rates
will be noisy and may tie at 0. We assert the MEASUREMENT exists, print the
direction honestly, and leave the conclusion to the real run.)*

In [ ]:
# ---- Experiment 1 · myth contamination A/B ----
RUNS = {}          # live models (clean arm is reused by Experiment 2)
EXP1 = None
try:
    EXP1, RUNS = xp.run_exp1(index, sets['mythtraps'], corpora=corpora,
                             mode=MODE, ckpt_root=CKPT_ROOT, tau=TAU)

    print('\n=== TABLE 1 · model x myth-parroting rate (RAW mode) ===')
    h, r = xp.myth_ab_table(EXP1)
    print(xp.md_table(h, r))

    print('\n--- context: per-mode parroting rates ---')
    h2, r2 = xp.myth_ab_modewise_table(EXP1)
    print(xp.md_table(h2, r2))

    print('\n--- example quotes of parroted answers (RAW) ---')
    any_ex = False
    for name in ('clean', 'contaminated'):
        for ex in EXP1['results'][name]['examples']:
            any_ex = True
            print('[%s · %s] %s' % (name, ex['id'], ex['question']))
            print('    -> %s' % ex['answer'])
    if not any_ex:
        print('(no trap was parroted by either brain in this run)')

    print('\nexpected direction (contaminated >= clean): %s'
          % ('HOLDS' if EXP1['expected_direction_holds']
             else 'does NOT hold in this run (noted honestly; noisy in smoke)'))
    xp.save_payload(EXP1, RESULTS / 'exp11_myth_ab.json')
    print('\nEXP1 DONE — myth contamination A/B measured (payload '
          'exp11_myth_ab.json)')
except Exception:
    traceback.print_exc()
    print('EXP1 FAILED — battery continues; summary.md will say so')

## 2 · Experiment 2 — grounding delta (the headline)

**Design.** Take the **clean** sibling from Experiment 1 and run the full
Task 9 harness: 3 eval sets × 4 modes, every final answer re-verified with
the SAME lexical yardstick (`eval_lib.evaluate_sets`).

| mode | what answers |
|------|--------------|
| RAW | the bare model (greedy, max 80 new tokens) |
| GROUNDED | answers assembled ONLY from retrieved corpus sentences, cited |
| VERIFIED | RAW claims checked; unsupported ones rewritten from evidence or flagged `[unsupported]` |
| ABSTAINING | grounded + refuses when the best top-1 BM25 score is below tau |

**The science:** the hallucination-rate drop from RAW → GROUNDED is the
project's headline result — *retrieval grounding as a hallucination
control, measured on a brain we trained ourselves.* The per-question
records are kept: Experiment 3 reuses them (the tau decision only needs
each question's top-1 BM25 score).

In [ ]:
# ---- Experiment 2 · grounding delta (3 sets x 4 modes) ----
EXP2 = None
try:
    clean_run = RUNS.get('clean')
    if clean_run is None:
        print('(E1 model not in memory — loading the clean checkpoint from disk)')
        clean_run = xp.load_trained(CKPT_ROOT / 'exp1_clean')
    if clean_run is None:
        raise RuntimeError('no clean model available — run Experiment 1 first '
                           '(or point CKPT_ROOT at a trained checkpoint dir)')

    EXP2, METRICS2, RECORDS2 = xp.run_exp2(index, sets, clean_run, mode=MODE,
                                            tau=TAU)

    print('\n=== TABLE 2 · grounding delta: 3 sets x 4 modes ===')
    h, r = xp.grounding_delta_table(METRICS2)
    print(xp.md_table(h, r))

    print('\neval_lib headline:', METRICS2['summary']['headline'])
    xp.save_payload(EXP2, RESULTS / 'exp11_grounding.json')
    print('\nEXP2 DONE — grounding delta measured on 3 sets x 4 modes '
          '(payload exp11_grounding.json)')
except Exception:
    traceback.print_exc()
    print('EXP2 FAILED — battery continues; summary.md will say so')

## 3 · Experiment 3 — abstention threshold sweep

**Design.** The ABSTAINING mode refuses a question when its best top-1 BM25
retrieval score is below tau. Task 9 calibrated tau **once** as the *median*
top-1 score over the 60 answerable eval questions (`ABSTAIN_TAU` =
20.0454) — a principled default, but a default, not a measurement.

Here tau is swept over 12 values (0.2× … 1.5× the calibrated median) and,
at each, we count:

- **true refusals** — unanswerables refused (refusing these is *correct*);
- **false refusals** — answerables refused (refusing these wastes the model);
- **precision / recall / F1** over the refusal decision.

**The science:** where is the actual F1-optimal operating point, and how far
is Task 9's hardcoded tau from it? (Note: the tau decision depends only on
retrieval scores, so the sweep reuses Experiment 2's per-question records —
no regeneration; a retrieval-only fallback exists if those are missing. The
sweep isolates the *tau-driven* component: grounded low-retrieval refusals
are tau-independent and excluded.)

In [ ]:
# ---- Experiment 3 · abstention threshold sweep ----
EXP3 = None
try:
    e2_payload = EXP2 or xp.load_payload(RESULTS / 'exp11_grounding.json')
    EXP3 = xp.run_exp3(index, sets['answerable'], sets['unanswerable'],
                       e2_payload=e2_payload, mode=MODE)

    print('\n=== TABLE 3 · tau sweep (refusal precision/recall/F1) ===')
    h, r = xp.tau_sweep_table(EXP3)
    print(xp.md_table(h, r))

    b = EXP3['best']
    f1_default = next((row['f1'] for row in EXP3['rows']
                       if abs(row['multiplier'] - 1.0) < 1e-9), None)
    print('\nOPERATING POINT (max F1): tau = %.3f (%.2fx calibrated median '
          '%.3f) · precision %.3f · recall %.3f · F1 %.3f'
          % (b['tau'], b['multiplier'], EXP3['calibrated_tau'],
             b['precision'], b['recall'], b['f1']))
    print('Task 9 hardcoded tau  : %.4f (F1 %.3f at 1.00x) — the sweep is the '
          'evidence-based refinement of that default'
          % (EXP3['hardcoded_tau'],
             f1_default if f1_default is not None else float('nan')))
    xp.save_payload(EXP3, RESULTS / 'exp11_tau_sweep.json')
    print('\nEXP3 DONE — tau sweep measured, operating point chosen '
          '(payload exp11_tau_sweep.json)')
except Exception:
    traceback.print_exc()
    print('EXP3 FAILED — battery continues; summary.md will say so')

## 4 · Experiment 4 — seed variance

**Design.** Train the **baby** config 3× on the clean corpus, seeds
42 / 7 / 1234, and measure RAW vs GROUNDED hallucination on the answerable
set for each. Report mean ± spread — that spread is the noise floor every
other number in this notebook should be read against.

**Baby-config choice (documented).** The brief says "Config A dims,
char-level or BPE-tiny — pick what trains in minutes". We pick **BPE-tiny**:
Config A's transformer dims (n_layer 2, n_head 4, d_model 128, d_ff 512,
ctx 256) with `vocab_size 4096`, so the ONE tokenizer (`corpus/merges.json`)
and the whole eval machinery apply unchanged — a char-level baby (vocab 96)
would need a second tokenizer + data pipeline for zero scientific gain.
Measured on the dev box: **~minutes per seed on CPU** (smoke shrinks the
dims to d_model 64 / ctx 128 and 60 steps/seed — path proof only).

**The science:** with n=3 seeds this is a rough range, not a confidence
interval — but if the RAW hallucination rate jumps around by ±0.1 across
seeds, then a single-seed "grounding helps by 0.05" claim would be inside
the noise.

In [ ]:
# ---- Experiment 4 · seed variance (baby config, 3 seeds) ----
EXP4 = None
try:
    EXP4, _rows = xp.run_exp4(index, sets['answerable'],
                              corpus_path=corpora['clean'], mode=MODE,
                              ckpt_root=CKPT_ROOT, tau=TAU, seeds=SEEDS)

    print('\n=== TABLE 4 · seed variance: RAW vs GROUNDED hallucination ===')
    h, r = xp.seed_variance_table(EXP4)
    print(xp.md_table(h, r))

    s = EXP4['summary']
    print('\nnoise floor: RAW %.3f ± %.3f · GROUNDED %.3f ± %.3f (%s)'
          % (s['raw_hallucination_rate']['mean'],
             s['raw_hallucination_rate']['spread'],
             s['grounded_hallucination_rate']['mean'],
             s['grounded_hallucination_rate']['spread'],
             s['spread_definition']))
    xp.save_payload(EXP4, RESULTS / 'exp11_seed_variance.json')
    print('\nEXP4 DONE — seed variance measured (%d seeds x RAW/GROUNDED, '
          'payload exp11_seed_variance.json)' % len(EXP4['rows']))
except Exception:
    traceback.print_exc()
    print('EXP4 FAILED — battery continues; summary.md will say so')

## 5 · `results/summary.md` — the artifact

One file, the shape Task 12's README will quote: **one-line answer** to the
research question, the **four tables**, **notes for non-experts**, and
**provenance** (config, steps, seeds, date, stub-or-real). Missing
experiments render as "(did not complete)" — the file never invents.

In [ ]:
# ---- write results/summary.md (the four tables + provenance) ----
RESULTS_PAYLOADS = {'exp1': EXP1, 'exp2': EXP2, 'exp3': EXP3, 'exp4': EXP4}
PROV = xp.provenance_block(
    mode=MODE, model_cfg=MODEL_CFG, baby_cfg=BABY_CFG,
    steps_ab=STEPS_AB, steps_seed=STEPS_SEED, seeds=SEEDS, seed_base=SEED_AB,
    corpora=('clean=health_qa.jsonl (%d pairs) · contaminated='
             'corpus_with_myths.jsonl (%d pairs, +%d planted myths)'
             % (n_clean, n_cont, n_cont - n_clean)),
    tau=TAU, results_dir=RESULTS, max_new_tokens=MAX_NEW_TOKENS,
    extra={'index chunks': str(index['N']),
           'eval sets': json.dumps({k: len(v) for k, v in sets.items()}),
           'python': sys.version.split()[0]})

SUMMARY_PATH = xp.write_summary_md(RESULTS / 'summary.md', mode=MODE,
                                   results=RESULTS_PAYLOADS, provenance=PROV)
print(SUMMARY_PATH)
print('\nsummary.md written to', RESULTS / 'summary.md')

### Interpretation guidance

- **Experiment 1** — a *higher* parroting rate for the contaminated sibling
  is the expected direction; a tie (or inversion) in a tiny smoke run is
  noise, not evidence. With the full battery, any move ≥ 1/20 traps (5 pts)
  is worth a sentence in the README; more is a finding.
- **Experiment 2** — expect RAW ≫ GROUNDED. If GROUNDED doesn't win, check
  `mean_best_sentence_f1` (retrieval quality) before doubting the idea.
- **Experiment 3** — the median-tau default maximizes *recall* by
  construction (it refuses ~half the answerables). The max-F1 point usually
  sits below it; moving the operating point is a one-line constant change.
- **Experiment 4** — treat any difference smaller than the seed spread as
  unproven. This table is the project's built-in skeptisim meter.
- Stub vs real: `LOCAL_SMOKE = True` runs prove the machinery; every
  conclusion in `summary.md` should be re-read after the real battery.

### DONE WHEN
- [ ] All four experiments printed their `EXPn DONE` lines (or an explicit
      traceback + `EXPn FAILED` that the summary acknowledges).
- [ ] `results/summary.md` exists with: one-line answer + 4 tables +
      non-expert notes + provenance incl. the `stub_or_real` flag.
- [ ] Re-run any cell after a kernel restart — checkpoints/payloads make it
      resume instead of retrain (`REUSED (no retrain)` lines).
- [ ] In Colab with `LOCAL_SMOKE = False`: the Config B A/B arms finish and
      the headline question is answered with real numbers.